# Acc-1: SSL pretrain (09 재실험)

**프로토콜:** `A_프로토콜/01_프로토콜_재설계.md`  
**출력:** `outputs/checkpoints/ssl_{tag}_seed{seed}_*.pt`  
**실험 조건:** `curriculum` | `none` | `fixed_emphasize` | `schedule_colorless` | `macenko_none`

Save & Run당 `CFG.ssl_mode`와 `CFG.seed` 하나만 바꿔 실행한다. Split은 `split_seed=42` 고정.
공유 `nct_split_seed42.json`을 Dataset으로 Add Input하면 `/kaggle/input`에서 찾아 복사한다. 없으면 새로 만든다.


## 0. 경로·임포트


In [ ]:
from __future__ import annotations

import json
import random
import shutil
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import amp
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from torchvision.transforms import functional as TF

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    PROJ_ROOT = Path("/kaggle/working/09_보완_재실험")
else:
    CWD = Path.cwd().resolve()
    PROJ_ROOT = None
    for c in [CWD, CWD.parent, CWD.parent.parent, CWD.parent.parent.parent]:
        if (c / "A_프로토콜").exists() or (c / "notebooks").exists() and (c / "outputs").exists():
            if c.name == "09_보완_재실험" or (c / "A_프로토콜").exists():
                PROJ_ROOT = c
                break
    if PROJ_ROOT is None:
        PROJ_ROOT = CWD

OUT = PROJ_ROOT / "outputs"
DIR_FIG = OUT / "figures"
DIR_MET = OUT / "metrics"
DIR_CKPT = OUT / "checkpoints"
DIR_SAMP = OUT / "samples"
DIR_LOG = OUT / "logs"
DIR_SPLIT = OUT / "splits"
for d in [DIR_FIG, DIR_MET, DIR_CKPT, DIR_SAMP, DIR_LOG, DIR_SPLIT]:
    d.mkdir(parents=True, exist_ok=True)

print("ON_KAGGLE:", ON_KAGGLE)
print("PROJ_ROOT:", PROJ_ROOT)
print("OUT:", OUT)


## 1. 데이터 헬퍼·설정


In [ ]:
NCT_CLASS_DIRS = ("ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM")

KAGGLE_NCT_CANDIDATES = [
    Path("/kaggle/input/nct-crc-he-100k-nonorm/NCT-CRC-HE-100K-NONORM"),
    Path("/kaggle/input/nct-crc-he-100k-nonorm"),
    Path("/kaggle/input/datasets/imrankhan77/nct-crc-he-100k-nonorm/NCT-CRC-HE-100K-NONORM"),
    Path("/kaggle/input/nct-crc-he-100k/NCT-CRC-HE-100K"),
    Path("/kaggle/input/nct-crc-he-100k"),
]
KAGGLE_VAL_CANDIDATES = [
    Path("/kaggle/input/crc-val-he-7k/CRC-VAL-HE-7K"),
    Path("/kaggle/input/crc-val-he-7k"),
    Path("/kaggle/input/datasets/imrankhan77/crc-val-he-7k/CRC-VAL-HE-7K"),
]


def _resolve_nct_folder(p: Path) -> Optional[Path]:
    if not p.is_dir():
        return None
    hits = sum(1 for c in NCT_CLASS_DIRS if (p / c).is_dir())
    if hits >= 3:
        return p
    for child in sorted(p.iterdir()):
        if child.is_dir():
            hits = sum(1 for c in NCT_CLASS_DIRS if (child / c).is_dir())
            if hits >= 3:
                return child
    return None


def find_root(cands) -> Optional[Path]:
    for cand in cands:
        r = _resolve_nct_folder(cand)
        if r is not None:
            return r
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    for p in sorted(root.rglob("*")):
        if not p.is_dir():
            continue
        r = _resolve_nct_folder(p)
        if r is None:
            continue
        low = str(p).lower()
        if cands is KAGGLE_VAL_CANDIDATES and ("val" in low or "7k" in low):
            return r
        if cands is KAGGLE_NCT_CANDIDATES and "val" not in low:
            return r
    return None


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class ImageFolderRGB(Dataset):
    def __init__(self, root: str, size: int, class_subset: Optional[List[str]] = None):
        self.base = datasets.ImageFolder(root)
        if class_subset:
            keep = {i for i, n in enumerate(self.base.classes) if n in class_subset}
            self.indices = [i for i, (_, y) in enumerate(self.base.samples) if y in keep]
            self.remap = {old: new for new, old in enumerate(sorted(keep))}
        else:
            self.indices = list(range(len(self.base)))
            self.remap = {i: i for i in range(len(self.base.classes))}
        self.size = size
        self.to_tensor = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor()])

    @property
    def num_classes(self) -> int:
        return len(self.remap)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, i: int):
        path, y = self.base.samples[self.indices[i]]
        from PIL import Image
        img = Image.open(path).convert("RGB")
        return self.to_tensor(img), self.remap[y]


def stratified_holdout_split(labels: List[int], val_frac: float, seed: int) -> Tuple[List[int], List[int]]:
    rng = np.random.RandomState(seed)
    by_cls: Dict[int, List[int]] = {}
    for i, y in enumerate(labels):
        by_cls.setdefault(int(y), []).append(i)
    train_idx, val_idx = [], []
    for y, idxs in sorted(by_cls.items()):
        idxs = list(idxs)
        rng.shuffle(idxs)
        n_val = max(1, int(round(len(idxs) * val_frac)))
        if n_val >= len(idxs):
            n_val = max(1, len(idxs) // 10)
        val_idx.extend(idxs[:n_val])
        train_idx.extend(idxs[n_val:])
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx


def adopt_shared_split_json(path: Path) -> None:
    if path.exists():
        return
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return
    hits = list(input_root.rglob(path.name))
    if not hits:
        print("no shared split in /kaggle/input; will create", path.name)
        return
    src = sorted(hits, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, path)
    print("copied shared split", src, "->", path)


def load_or_create_split(full: ImageFolderRGB, max_images: int, val_frac: float, split_seed: int) -> dict:
    path = DIR_SPLIT / f"nct_split_seed{split_seed}.json"
    adopt_shared_split_json(path)
    labels = [full.remap[full.base.samples[full.indices[i]][1]] for i in range(len(full))]
    pool = list(range(len(full)))
    rng = random.Random(split_seed)
    rng.shuffle(pool)
    pool = pool[: min(max_images, len(pool))]
    pool_labels = [labels[i] for i in pool]
    if path.exists():
        split = json.loads(path.read_text(encoding="utf-8"))
        print("loaded split", path)
        return split
    local_train, local_val = stratified_holdout_split(pool_labels, val_frac, split_seed)
    train_idx = [pool[i] for i in local_train]
    val_idx = [pool[i] for i in local_val]
    split = {
        "split_seed": split_seed,
        "val_frac": val_frac,
        "max_images": max_images,
        "n_pool": len(pool),
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "train_idx": train_idx,
        "val_idx": val_idx,
        "note": "CRC-VAL must never appear in this file",
    }
    path.write_text(json.dumps(split, ensure_ascii=False, indent=2), encoding="utf-8")
    print("wrote split", path, "train", len(train_idx), "val", len(val_idx))
    return split


def subsample_by_frac(indices: List[int], labels: List[int], frac: float, seed: int) -> List[int]:
    if frac >= 1.0:
        return list(indices)
    rng = np.random.RandomState(seed)
    by_cls: Dict[int, List[int]] = {}
    for i, y in zip(indices, labels):
        by_cls.setdefault(int(y), []).append(i)
    out = []
    for y, idxs in sorted(by_cls.items()):
        idxs = list(idxs)
        rng.shuffle(idxs)
        n = max(1, int(round(len(idxs) * frac)))
        out.extend(idxs[:n])
    rng.shuffle(out)
    return out


_DETECTED_NCT = find_root(KAGGLE_NCT_CANDIDATES)
print("NCT:", _DETECTED_NCT)


@dataclass
class TrainConfig:
    seed: int = 42
    split_seed: int = 42
    val_frac: float = 0.10
    ssl_mode: str = "curriculum"  # curriculum|none|fixed_emphasize|schedule_colorless|macenko_none
    data_root: str = field(default_factory=lambda: str(_DETECTED_NCT) if _DETECTED_NCT else "")
    image_size: int = 224
    max_train_images: int = 50000
    num_classes_cap: int = 9
    batch_size: int = 128
    num_workers: int = 2
    ssl_epochs: int = 40
    lr_ssl: float = 3e-4
    weight_decay: float = 1e-4
    temperature: float = 0.2
    proj_dim: int = 128
    color_emphasize_max: float = 0.85
    color_jitter_max: float = 0.85
    ckpt_every: int = 5
    resume_path: str = ""
    use_amp: bool = True
    backbone: str = "resnet18"
    use_macenko: bool = False
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")


CFG = TrainConfig()
# --- Run switches ---
# CFG.ssl_mode = "curriculum"
# CFG.ssl_mode = "none"
# CFG.ssl_mode = "fixed_emphasize"
# CFG.ssl_mode = "schedule_colorless"
# CFG.ssl_mode = "macenko_none"
# CFG.seed = 42  # or 43, 44
CFG.use_macenko = CFG.ssl_mode == "macenko_none"

set_seed(CFG.seed)
DEVICE = torch.device(CFG.device)
assert CFG.data_root and Path(CFG.data_root).exists(), "Add NCT NONORM input"
print(json.dumps(asdict(CFG), indent=2))


## 2. 증강·모델


In [ ]:
def rgb_to_hsv_torch(x: torch.Tensor) -> torch.Tensor:
    squeeze = False
    if x.ndim == 3:
        x = x.unsqueeze(0)
        squeeze = True
    r, g, b = x[:, 0], x[:, 1], x[:, 2]
    maxc, _ = x.max(dim=1)
    minc, _ = x.min(dim=1)
    v = maxc
    deltac = maxc - minc
    s = torch.where(maxc > 0, deltac / (maxc + 1e-8), torch.zeros_like(maxc))
    rc = (maxc - r) / (deltac + 1e-8)
    gc = (maxc - g) / (deltac + 1e-8)
    bc = (maxc - b) / (deltac + 1e-8)
    h = torch.zeros_like(maxc)
    h = torch.where((maxc == r) & (deltac > 0), (bc - gc), h)
    h = torch.where((maxc == g) & (deltac > 0), 2.0 + (rc - bc), h)
    h = torch.where((maxc == b) & (deltac > 0), 4.0 + (gc - rc), h)
    h = (h / 6.0) % 1.0
    hsv = torch.stack([h, s, v], dim=1)
    return hsv.squeeze(0) if squeeze else hsv


def hsv_to_rgb_torch(hsv: torch.Tensor) -> torch.Tensor:
    squeeze = False
    if hsv.ndim == 3:
        hsv = hsv.unsqueeze(0)
        squeeze = True
    h, s, v = hsv[:, 0], hsv[:, 1], hsv[:, 2]
    h6 = h * 6.0
    i = torch.floor(h6).long() % 6
    f = h6 - torch.floor(h6)
    p = v * (1.0 - s)
    q = v * (1.0 - f * s)
    t = v * (1.0 - (1.0 - f) * s)
    out = torch.zeros_like(hsv)
    for idx, (rr, gg, bb) in enumerate([(v, t, p), (q, v, p), (p, v, t), (p, q, v), (t, p, v), (v, p, q)]):
        mask = i == idx
        out[:, 0][mask] = rr[mask]
        out[:, 1][mask] = gg[mask]
        out[:, 2][mask] = bb[mask]
    return out.squeeze(0) if squeeze else out


def emphasize_color(x: torch.Tensor, sat_gain: float = 1.6, contrast: float = 1.15) -> torch.Tensor:
    single = x.ndim == 3
    if single:
        x = x.unsqueeze(0)
    hsv = rgb_to_hsv_torch(x.clamp(0, 1)).clone()
    hsv[:, 1] = (hsv[:, 1] * sat_gain).clamp(0, 1)
    rgb = hsv_to_rgb_torch(hsv)
    mean = rgb.mean(dim=(-2, -1), keepdim=True)
    rgb = (mean + (rgb - mean) * contrast).clamp(0, 1)
    return rgb.squeeze(0) if single else rgb


def curriculum_probs(epoch: int, total: int, cfg: TrainConfig) -> Tuple[float, float]:
    t = 0.0 if total <= 1 else epoch / (total - 1)
    return cfg.color_emphasize_max * (1.0 - t), cfg.color_jitter_max * t


def make_ssl_views(x: torch.Tensor, p_emp: float, p_jit: float, fixed_emphasize: bool = False):
    def one():
        out = x
        if fixed_emphasize or random.random() < p_emp:
            out = emphasize_color(out)
        if (not fixed_emphasize) and random.random() < p_jit:
            out = TF.adjust_hue(out, random.uniform(-0.05, 0.05))
            out = TF.adjust_saturation(out, 1.0 + random.uniform(-0.3, 0.3) * p_jit)
        if random.random() < 0.5:
            out = TF.hflip(out)
        if (not fixed_emphasize) and random.random() < (0.15 + 0.35 * p_jit):
            out = TF.gaussian_blur(out.unsqueeze(0), kernel_size=3).squeeze(0)
        return out.clamp(0, 1)
    return one(), one()


def make_ssl_views_colorless(x: torch.Tensor, t: float):
    """P1-1: schedule without color emphasis / hue-sat jitter."""
    def one():
        out = x
        strength = 0.1 + 0.5 * t
        if random.random() < 0.8:
            out = TF.adjust_brightness(out, 1.0 + random.uniform(-strength, strength))
            out = TF.adjust_contrast(out, 1.0 + random.uniform(-strength, strength))
        if random.random() < 0.5:
            out = TF.hflip(out)
        # random resized crop via crop+resize approximation
        _, h, w = out.shape
        scale = random.uniform(0.2 + 0.6 * t, 1.0)
        ch, cw = max(8, int(h * scale)), max(8, int(w * scale))
        top = random.randint(0, max(0, h - ch))
        left = random.randint(0, max(0, w - cw))
        out = TF.resized_crop(out, top, left, ch, cw, [h, w])
        if random.random() < (0.15 + 0.35 * t):
            out = TF.gaussian_blur(out.unsqueeze(0), kernel_size=3).squeeze(0)
        return out.clamp(0, 1)
    return one(), one()


def try_macenko(x: torch.Tensor) -> torch.Tensor:
    """Lightweight stain re-centering fallback if full Macenko unavailable."""
    # Approximate: channel-wise standardize then rescale to [0,1]
    mean = x.mean(dim=(-2, -1), keepdim=True)
    std = x.std(dim=(-2, -1), keepdim=True).clamp_min(1e-4)
    y = (x - mean) / std
    y = y * 0.2 + 0.5
    return y.clamp(0, 1)


class Encoder(nn.Module):
    def __init__(self, backbone: str = "resnet18"):
        super().__init__()
        net = models.resnet18(weights=None)
        feat = net.fc.in_features
        net.fc = nn.Identity()
        self.net = net
        self.out_dim = feat

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int, proj_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim), nn.ReLU(inplace=True), nn.Linear(in_dim, proj_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def nt_xent(z1: torch.Tensor, z2: torch.Tensor, temperature: float) -> torch.Tensor:
    z1 = F.normalize(z1.float(), dim=1)
    z2 = F.normalize(z2.float(), dim=1)
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = z @ z.t() / temperature
    mask = torch.eye(2 * B, device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(mask, torch.finfo(sim.dtype).min)
    pos = torch.cat([torch.arange(B, 2 * B), torch.arange(0, B)]).to(z.device)
    return F.cross_entropy(sim, pos)


## 3. Split·로더 (hold-out val 제외, SSL은 train pool만)


In [ ]:
probe = datasets.ImageFolder(CFG.data_root)
preferred = [c for c in NCT_CLASS_DIRS if c in probe.classes][: CFG.num_classes_cap]
full = ImageFolderRGB(CFG.data_root, CFG.image_size, class_subset=preferred)
split = load_or_create_split(full, CFG.max_train_images, CFG.val_frac, CFG.split_seed)
# SSL uses FT-train indices only (never hold-out val, never CRC-VAL)
ssl_ds = Subset(full, split["train_idx"])
train_loader = DataLoader(
    ssl_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)
NUM_CLASSES = full.num_classes
print("ssl_pool:", len(ssl_ds), "holdout_val:", split["n_val"], "classes:", preferred)


## 4. SSL 학습


In [ ]:
enc = Encoder(CFG.backbone).to(DEVICE)
proj = ProjectionHead(enc.out_dim, CFG.proj_dim).to(DEVICE)
opt = torch.optim.AdamW(
    list(enc.parameters()) + list(proj.parameters()), lr=CFG.lr_ssl, weight_decay=CFG.weight_decay
)
scaler = amp.GradScaler("cuda", enabled=CFG.use_amp and DEVICE.type == "cuda")
tag = CFG.ssl_mode
rows = []
start_ep = 0

if CFG.resume_path and Path(CFG.resume_path).exists():
    ckpt = torch.load(CFG.resume_path, map_location=DEVICE)
    enc.load_state_dict(ckpt["encoder"])
    proj.load_state_dict(ckpt["proj"])
    if "optimizer" in ckpt:
        opt.load_state_dict(ckpt["optimizer"])
    start_ep = int(ckpt.get("epoch", -1)) + 1
    rows = list(ckpt.get("log_rows", []))
    print("resumed", CFG.resume_path, "next_ep", start_ep)

fixed = CFG.ssl_mode == "fixed_emphasize"
t0 = time.time()
for ep in range(start_ep, CFG.ssl_epochs):
    t = 0.0 if CFG.ssl_epochs <= 1 else ep / (CFG.ssl_epochs - 1)
    if CFG.ssl_mode == "curriculum":
        p_emp, p_jit = curriculum_probs(ep, CFG.ssl_epochs, CFG)
    elif CFG.ssl_mode == "fixed_emphasize":
        p_emp, p_jit = 1.0, 0.0
    elif CFG.ssl_mode in ("none", "macenko_none"):
        p_emp, p_jit = 0.0, 0.35
    else:  # schedule_colorless
        p_emp, p_jit = 0.0, t

    enc.train()
    proj.train()
    losses = []
    for x, _ in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        if CFG.use_macenko:
            x = torch.stack([try_macenko(x[i]) for i in range(x.size(0))])
        views = []
        for i in range(x.size(0)):
            if CFG.ssl_mode == "schedule_colorless":
                v1, v2 = make_ssl_views_colorless(x[i], t)
            else:
                v1, v2 = make_ssl_views(x[i], p_emp, p_jit, fixed)
            views.append((v1, v2))
        v1 = torch.stack([a for a, _ in views])
        v2 = torch.stack([b for _, b in views])
        opt.zero_grad(set_to_none=True)
        with amp.autocast("cuda", enabled=CFG.use_amp and DEVICE.type == "cuda"):
            loss = nt_xent(proj(enc(v1)), proj(enc(v2)), CFG.temperature)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        losses.append(float(loss.item()))

    row = {
        "epoch": ep,
        "loss": float(np.mean(losses)),
        "p_emp": float(p_emp),
        "p_jit": float(p_jit),
        "t": float(t),
        "ssl_mode": tag,
        "seed": CFG.seed,
        "hours": (time.time() - t0) / 3600.0,
    }
    rows.append(row)
    print(row)
    payload = {
        "encoder": enc.state_dict(),
        "proj": proj.state_dict(),
        "optimizer": opt.state_dict(),
        "scaler": scaler.state_dict() if scaler is not None else None,
        "epoch": ep,
        "ssl_mode": tag,
        "seed": CFG.seed,
        "config": asdict(CFG),
        "log_rows": rows,
        "split_path": str(DIR_SPLIT / f"nct_split_seed{CFG.split_seed}.json"),
    }
    last_path = DIR_CKPT / f"ssl_{tag}_seed{CFG.seed}_last.pt"
    torch.save(payload, last_path)
    if (ep + 1) % CFG.ckpt_every == 0 or ep + 1 == CFG.ssl_epochs:
        torch.save(payload, DIR_CKPT / f"ssl_{tag}_seed{CFG.seed}_ep{ep:03d}.pt")

final_path = DIR_CKPT / f"ssl_{tag}_seed{CFG.seed}_final.pt"
torch.save(payload, final_path)
pd.DataFrame(rows).to_csv(DIR_MET / f"ssl_log_{tag}_seed{CFG.seed}.csv", index=False)
handoff = {
    "ssl_mode": tag,
    "seed": CFG.seed,
    "final_ckpt": str(final_path),
    "n_ssl": len(ssl_ds),
    "split_seed": CFG.split_seed,
}
(DIR_MET / f"handoff_{tag}_seed{CFG.seed}.json").write_text(
    json.dumps(handoff, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(handoff, indent=2))


## 체크리스트
1. NCT NONORM Add Input  
2. `CFG.ssl_mode` / `CFG.seed` 설정  
3. Save & Run All → `outputs/checkpoints/ssl_*_final.pt`  
4. Acc-2에서 해당 ckpt Add Input
